In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers scikit-learn lxml
!pip install datasets


In [ ]:
import os

INPUT_FILE  = "/kaggle/input/datasets/moulighosh28/hallucination-classifier/hallucination_dataset2 (4).npz"
save_file = "/kaggle/working/hallucination_dataset2.npz"

In [ ]:
from datasets import load_dataset

data = []

dataset = load_dataset("medalpaca/medical_meadow_medical_flashcards")

train_data = dataset["train"]
print("Total rows in dataset:", len(train_data))

print("Columns:", train_data.column_names)

subset = train_data.select(range(1, 25000))

prev_question = None   # keep track of previous question

for row in subset:

    question = row["input"].strip()
    answer = row["output"].strip()

    # skip if same as previous question
    if question == prev_question:
        continue

    if question and answer:
        data.append((question, answer))
        prev_question = question   # update previous question

print("Total valid Q/A pairs:", len(data))

In [ ]:

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.3"

# Quantization config (NEW WAY)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    output_hidden_states=True,
    output_attentions=True
)

model.eval()


In [ ]:
import torch
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import AutoModelForSequenceClassification, AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#  Similarity model
sim_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)  #SBERT

#  NLI model
nli_name = "roberta-large-mnli"

nli_tokenizer = AutoTokenizer.from_pretrained(nli_name)

nli_model = AutoModelForSequenceClassification.from_pretrained(nli_name)
nli_model.to(device)
nli_model.eval()


In [ ]:
def compute_entropy(probs):
    probs = torch.clamp(probs, min=1e-9)
    return -(probs * torch.log(probs)).sum(dim=-1)


def compute_similarity(ref, gen):
    with torch.no_grad():
        emb = sim_model.encode([ref, gen], convert_to_tensor=True)
        score = util.cos_sim(emb[0], emb[1])
    return score.item()


def run_nli(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(nli_model.device)

    with torch.no_grad():
        outputs = nli_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)
    labels = ["contradiction", "neutral", "entailment"]

    return labels[torch.argmax(probs, dim=-1).item()]

In [ ]:
def extract_first_token_features(prompt, model, tokenizer, top_k=20):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    input_ids = inputs["input_ids"]
    start_pos = input_ids.shape[1]

    # ── First forward pass — get next token ──────────────────
    with torch.no_grad():
        outputs    = model(**inputs)
        logits     = outputs.logits[:, -1, :]
        next_token = torch.argmax(logits, dim=-1).unsqueeze(-1)

    full_input_ids = torch.cat([input_ids, next_token], dim=1)

    # ── Second forward pass — get internals ──────────────────
    with torch.no_grad():
        outputs = model(
            input_ids=full_input_ids,
            output_hidden_states=True,
            output_attentions=True
        )

    # 1. SOFTMAX FEATURES ─────────────────────────────────────
    logits_first     = outputs.logits[:, start_pos - 1, :]
    probs            = torch.softmax(logits_first, dim=-1)
    topk_vals, _     = torch.topk(probs, k=top_k, dim=-1)
    softmax_features = topk_vals.squeeze(0).float().cpu().numpy()
    # shape: [20] ✅

    # 2. ATTENTION FEATURES ───────────────────────────────────
    MAX_INPUT_LEN = 64
    attn_last = outputs.attentions[-1]  # [1, num_heads, seq_len, seq_len]
    num_heads = attn_last.shape[1]      # 32 for Mistral-7B

    # attention row of first generated token over all input tokens
    attn_row   = attn_last[0, :, start_pos, :start_pos]
    # shape: [32, actual_input_len]

    actual_len = min(attn_row.shape[1], MAX_INPUT_LEN)

    # Padded — fixed size, created on same device as model
    attn_fixed = torch.zeros(num_heads, MAX_INPUT_LEN,
                             dtype=torch.float32,
                             device=attn_last.device)  # ← same device ✅
    attn_fixed[:, :actual_len] = attn_row[:, :actual_len].float()
    attn_padded = attn_fixed.cpu().numpy().flatten()
    # shape: [2048] ✅

    # Raw — variable size for flexible later use
    attn_raw = attn_row.detach().float().cpu().numpy().flatten()
    # shape: [32 * actual_len] — variable ✅

    # 3. FFN FEATURES ─────────────────────────────────────────
    last_layer        = model.model.layers[-1]
    hidden_before_ffn = outputs.hidden_states[-2][:, start_pos, :]
    normed_hidden     = last_layer.post_attention_layernorm(hidden_before_ffn)
    gate              = last_layer.mlp.gate_proj(normed_hidden)
    up                = last_layer.mlp.up_proj(normed_hidden)
    ffn_activation    = torch.nn.functional.silu(gate) * up
    ffn_vector        = ffn_activation.squeeze(0).detach().float().cpu().numpy()
    # shape: [14336] ✅

    return softmax_features, attn_padded, attn_raw, ffn_vector

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"# Initialize
import gc
from tqdm import tqdm
session_count = 0
X_softmax     = []
X_attn_padded = []  # fixed [2048]
X_attn_raw    = []  # variable
X_ffn         = []
labels        = []

# Load checkpoint
if os.path.exists(INPUT_FILE):
    saved = np.load(INPUT_FILE, allow_pickle=True)
    X_softmax     = list(saved["X_softmax"])
    X_attn_padded = list(saved["X_attn_padded"])
    X_attn_raw    = list(saved["X_attn_raw"])
    X_ffn         = list(saved["X_ffn"])
    labels        = list(saved["y"])
    start_index = int(saved["last_index"]) + 1
    print(f"Resumed from checkpoint: {start_index} samples")
else:
    start_index = 0

print("Loaded samples:", len(labels))
tokenizer.pad_token = tokenizer.eos_token

# Main loop
for i in tqdm(range(start_index, len(data))):
    session_count += 1

    if i % 50 == 0:
        gc.collect()
        torch.cuda.empty_cache()

    question, reference = data[i]
    prompt = f"Answer the following question in one short sentence \nQuestion: {question}\nAnswer:"

    # Extract features
    with torch.no_grad():
        softmax_f, attn_padded_f, attn_raw_f, ffn_f = \
            extract_first_token_features(prompt, model, tokenizer)

    # Generate full answer for labeling
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=64
    ).to(model.device)

    with torch.no_grad():
        gen_ids = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,
            use_cache=False
        )

    input_len  = inputs["attention_mask"].sum().item()
    gen_tokens = gen_ids[0][int(input_len):]
    generated  = tokenizer.decode(gen_tokens, skip_special_tokens=True)



    # Labeling
    similarity = compute_similarity(reference, generated)
    nli_result = run_nli(reference, generated)

    # Add safety check: if NLI and similarity strongly disagree, skip or flag for review
    if nli_result == "entailment" and similarity < 0.60:
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()  
        continue  # or label = -1 for manual review
    elif nli_result == "contradiction" and similarity > 0.85: 
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()
        continue
    elif nli_result == "entailment":
        label = 0
    elif nli_result == "contradiction":
        label = 1
    elif similarity >= 0.91:
        label = 0
    elif similarity <= 0.40:
        label = 1
    else:
        del inputs, gen_ids, gen_tokens
        del softmax_f, attn_padded_f, attn_raw_f, ffn_f  # ✅
        gc.collect()
        torch.cuda.empty_cache()
        continue

   



    # Append
    X_softmax.append(softmax_f)
    X_attn_padded.append(attn_padded_f)  # [2048]
    X_attn_raw.append(attn_raw_f)        # variable
    X_ffn.append(ffn_f)
    labels.append(label)

    if session_count < 10 :
        print("\nQUESTION:", question)
        print("REFERENCE:", reference)
        print("GENERATED:", generated)
        print("-" * 10)
        print("Similarity:", similarity)
        print("NLI:", nli_result)
        print("Assigned label:", label)
        print("-" * 40)

    if len(labels) % 50 == 0:
        print("\nQUESTION:", question)
        print("REFERENCE:", reference)
        print("GENERATED:", generated)
        print("-" * 10)
        print("Similarity:", similarity)
        print("NLI:", nli_result)
        print("Assigned label:", label)
        print("-" * 40)

    # Cleanup
    del inputs, gen_ids, gen_tokens
    del softmax_f, attn_padded_f, attn_raw_f, ffn_f
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    # Checkpoint every 20 samples
    if len(labels) % 20 == 0:
        np.savez(
            save_file,
            X_softmax     = np.array(X_softmax),
            X_attn_padded = np.array(X_attn_padded),
            X_attn_raw    = np.array(X_attn_raw, dtype=object),
            X_ffn         = np.array(X_ffn),
            y             = np.array(labels),
            last_index    = i
        )
        print("Checkpoint saved at", len(labels))

# Final save
y = np.array(labels)
print("Label 0:", np.sum(y == 0))
print("Label 1:", np.sum(y == 1))
print("Final dataset size:", len(y))

np.savez(
    save_file,
    X_softmax     = np.array(X_softmax),
    X_attn_padded = np.array(X_attn_padded),
    X_attn_raw    = np.array(X_attn_raw, dtype=object),
    X_ffn         = np.array(X_ffn),
    y             = y,
    last_index    = i
)
print("Final dataset saved.")

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# ---------------------------------
# Load from checkpoint
# ---------------------------------

save_file = "/content/drive/MyDrive/hallucination_dataset1.npz"

saved         = np.load(save_file, allow_pickle=True)
X_softmax     = np.array(list(saved["X_softmax"]))
X_attn_padded = np.array(list(saved["X_attn_padded"]))
X_ffn         = np.array(list(saved["X_ffn"]))
y             = np.array(list(saved["y"]))

print(f"Loaded {len(y)} samples")
print(f"Label 0 (non-hallucinated): {np.sum(y == 0)}")
print(f"Label 1 (hallucinated):     {np.sum(y == 1)}")

# ---------------------------------
# SPLIT (same indices for all)
# ---------------------------------

indices = np.arange(len(y))

train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=42
)

def prepare_data(X, scaler_type="standard"):
    X_train = X[train_idx]
    X_test  = X[test_idx]

    if scaler_type == "minmax":
        scaler = MinMaxScaler()
    else:
        scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test  = scaler.transform(X_test)

    return X_train, X_test

# Prepare each feature set
X_softmax_train, X_softmax_test = prepare_data(X_softmax,     "standard")
X_attn_train,    X_attn_test    = prepare_data(X_attn_padded, "minmax")
X_ffn_train,     X_ffn_test     = prepare_data(X_ffn,         "standard")

y_train = y[train_idx]
y_test  = y[test_idx]

# ---------------------------------
# Device
# ---------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def to_tensor(X, y):
    return (
        torch.tensor(X, dtype=torch.float32).to(device),
        torch.tensor(y, dtype=torch.long).to(device)
    )

X_softmax_train, y_train_t = to_tensor(X_softmax_train, y_train)
X_softmax_test,  y_test_t  = to_tensor(X_softmax_test,  y_test)

X_attn_train, _ = to_tensor(X_attn_train, y_train)
X_attn_test,  _ = to_tensor(X_attn_test,  y_test)

X_ffn_train, _  = to_tensor(X_ffn_train,  y_train)
X_ffn_test,  _  = to_tensor(X_ffn_test,   y_test)

# ---------------------------------
# Class weights
# ---------------------------------

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

# ---------------------------------
# Model
# ---------------------------------

class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

# ---------------------------------
# Train function
# ---------------------------------

def train_model(X_train, y_train, input_dim):
    model     = Classifier(input_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    batch_size = 128
    epochs     = 50

    for epoch in range(epochs):
        model.train()
        perm       = torch.randperm(X_train.size(0))
        epoch_loss = 0

        for i in range(0, X_train.size(0), batch_size):
            idx     = perm[i:i + batch_size]
            x_batch = X_train[idx]
            y_batch = y_train[idx]

            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        if epoch % 10 == 0:
            print(f"Epoch {epoch} | Loss: {epoch_loss:.4f}")

    return model

# ---------------------------------
# Train 3 classifiers
# ---------------------------------

print("\nTraining SOFTMAX model")
clf_softmax = train_model(X_softmax_train, y_train_t, X_softmax_train.shape[1])

print("\nTraining ATTENTION model")
clf_attn = train_model(X_attn_train, y_train_t, X_attn_train.shape[1])

print("\nTraining FFN model")
clf_ffn = train_model(X_ffn_train, y_train_t, X_ffn_train.shape[1])

# ---------------------------------
# Evaluation (ensemble)
# ---------------------------------

def get_probs(model, X):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(X), dim=1)[:, 1]

p_softmax = get_probs(clf_softmax, X_softmax_test)
p_attn    = get_probs(clf_attn,    X_attn_test)
p_ffn     = get_probs(clf_ffn,     X_ffn_test)

# Weighted ensemble
probs = (
    0.2 * p_softmax +
    0.3 * p_attn    +
    0.5 * p_ffn
)

roc   = roc_auc_score(y_test, probs.cpu())
preds = (probs > 0.5).cpu()
f1    = f1_score(y_test, preds)

print("\n--- Results ---")
print(f"ROC-AUC:  {roc:.4f}")
print(f"F1 Score: {f1:.4f}")